TaaSim — Exploration NYC TLC Trip Records

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import pandas as pd

verifier la connexion au jupyter docker 

In [2]:
import os
print(os.getcwd())

/home/jovyan


In [4]:


# 1. Créer la session Spark connectée à MinIO
spark = SparkSession.builder \
    .appName("TaaSim-NYC-Exploration") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 3.5.0


2. Charger 1 mois NYC TLC

In [ ]:
# Chemin dans MinIO après upload manuel 
df = spark.read.parquet("s3a://raw/nyc-tlc/yellow_tripdata_2025-01.parquet")
# Vue  du schéma
df.printSchema()

3. Statistiques de base — combien de lignes ? valeurs nulles ?

In [ ]:
total = df.count()
print(f"Total lignes : {total:,}")

# Compter les nulls par colonne
from pyspark.sql.functions import col, sum as _sum, isnan, when

null_counts = df.select([
    _sum(when(col(c).isNull() | isnan(col(c)), 1).otherwise(0)).alias(c)
    for c in df.columns
])
null_counts.show(vertical=True)


4. Nettoyage des anomalies (crucial pour le ML plus tard)

In [ ]:
df_clean = df.filter(
    (F.col("trip_distance") > 0) &          # pas de trajet à 0 km
    (F.col("trip_distance") < 100) &         # pas de trajets absurdes
    (F.col("fare_amount") > 0) &             # pas de tarifs négatifs
    (F.col("fare_amount") < 500) &           # pas de tarifs aberrants
    (F.col("passenger_count") > 0) &         # au moins 1 passager
    (F.col("passenger_count") <= 6) &        # max taxi standard
    (F.col("tpep_pickup_datetime").isNotNull()) &
    (F.col("PULocationID").between(1, 265))  # zones valides NYC
)

removed = total - df_clean.count()
print(f"Lignes supprimées : {removed:,} ({removed/total*100:.1f}%)")

 5. Distribution de la demande par heure 

In [ ]:
hourly_demand = df_clean \
    .withColumn("hour", F.hour("tpep_pickup_datetime")) \
    .groupBy("hour") \
    .agg(F.count("*").alias("nb_trips")) \
    .orderBy("hour")

# Convertir en Pandas pour visualisation
hourly_pd = hourly_demand.toPandas()

plt.figure(figsize=(12, 4))
plt.bar(hourly_pd["hour"], hourly_pd["nb_trips"], color="#378ADD", alpha=0.8)
plt.xlabel("Heure de la journée")
plt.ylabel("Nombre de trajets")
plt.title("Distribution de la demande par heure — NYC TLC Janvier 2023")
plt.xticks(range(0, 24))
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("nyc_hourly_demand.png", dpi=150)
plt.show()

# Identifier les heures de pointe
peak_hours = hourly_pd.nlargest(5, "nb_trips")
print("Top 5 heures de pointe:")
print(peak_hours.to_string(index=False))

6.Top 10 zones de pickup

In [ ]:
top_zones = df_clean \
    .groupBy("PULocationID") \
    .agg(F.count("*").alias("nb_trips")) \
    .orderBy(F.desc("nb_trips")) \
    .limit(10)

top_zones.show()

# Pour TaaSim : ces zones = candidats pour les zones à forte demande
# Dans Casablanca, l'équivalent sera : Maarif, Ain Diab, Casa-Port...

 7. Statistiques des trajets — durée, distance, tarif

In [ ]:
df_with_duration = df_clean.withColumn(
    "trip_duration_min",
    (F.unix_timestamp("tpep_dropoff_datetime") -
     F.unix_timestamp("tpep_pickup_datetime")) / 60
).filter(
    (F.col("trip_duration_min") > 1) &   # > 1 minute
    (F.col("trip_duration_min") < 120)   # < 2 heures
)

df_with_duration.select(
    "trip_distance",
    "trip_duration_min",
    "fare_amount",
    "passenger_count"
).describe().show()

8. Pattern jour de semaine — important pour les features ML

In [ ]:
daily_demand = df_clean \
    .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime")) \
    .withColumn("day_name", 
        F.when(F.col("day_of_week") == 1, "Dimanche")
         .when(F.col("day_of_week") == 2, "Lundi")
         .when(F.col("day_of_week") == 3, "Mardi")
         .when(F.col("day_of_week") == 4, "Mercredi")
         .when(F.col("day_of_week") == 5, "Jeudi")
         .when(F.col("day_of_week") == 6, "Vendredi")
         .otherwise("Samedi")
    ) \
    .groupBy("day_of_week", "day_name") \
    .agg(F.count("*").alias("nb_trips")) \
    .orderBy("day_of_week")

daily_demand.show()

 9.Agréger par zone + heure → structure cible pour le ML

In [ ]:
# C'est exactement le format qu'on va produire dans curated/demand-by-zone/

zone_hour_demand = df_clean \
    .withColumn("hour", F.hour("tpep_pickup_datetime")) \
    .withColumn("date", F.to_date("tpep_pickup_datetime")) \
    .groupBy("date", "hour", "PULocationID") \
    .agg(
        F.count("*").alias("nb_trips"),
        F.avg("trip_distance").alias("avg_distance_km"),
        F.avg("fare_amount").alias("avg_fare")
    ) \
    .orderBy("date", "hour", "PULocationID")

print(f"Nombre de combinaisons zone×heure : {zone_hour_demand.count():,}")
zone_hour_demand.show(10)

# Écrire en Parquet dans MinIO curated/
zone_hour_demand.write \
    .mode("overwrite") \
    .partitionBy("date") \
    .parquet("s3a://curated/demand-by-zone/")

print("Écrit dans MinIO curated/demand-by-zone/")